# DoorDash Geo-Hunt — GPU Pipeline (Colab T4)

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# 1. Clone repo + install
!git clone https://github.com/Eswar-deep/doordash-geo-hunt.git
%cd doordash-geo-hunt
!git checkout cursor/contest-day-parallel-8caa
!pip install -e . -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — change runtime!'}")

In [ ]:
# 2. Set API keys (fill these in!)
import os
os.environ["VISION_LLM_PROVIDER"] = "bedrock"
os.environ["GOOGLE_MAPS_API_KEY"] = ""  # FILL
os.environ["AWS_BEARER_TOKEN_BEDROCK"] = ""  # FILL
os.environ["AWS_BEDROCK_REGION"] = "us-east-1"
os.environ["AWS_BEDROCK_SONNET_MODEL_ID"] = "us.anthropic.claude-sonnet-4-6"
os.environ["AWS_BEDROCK_OPUS_MODEL_ID"] = "us.anthropic.claude-opus-4-6"
os.environ["AWS_BEDROCK_OPUS_FALLBACK_MODEL_ID"] = "us.anthropic.claude-opus-4-5-20251101-v1:0"

# ViT-H-14 on T4 GPU (best accuracy)
os.environ["CLIP_MODEL_NAME"] = "ViT-H-14"
os.environ["CLIP_PRETRAINED"] = "laion2b_s32b_b79k"

In [ ]:
# 3. Prewarm (downloads ViT-H-14 weights ~3.9GB, one-time per session)
!python cli.py prewarm

In [ ]:
# 4. RUN A DROP — paste tweet URL below
TWEET_URL = "https://x.com/DoorDash/status/PASTE_ID_HERE"

!python cli.py ingest "{TWEET_URL}" --out samples/live-drop --run --tweet-id \
  --agents streetview,vlm --staged --staged-parallel --sv-workers 32

In [ ]:
# 5. View results
import json
from pathlib import Path

jsons = sorted(Path("output").glob("*.json"))
if jsons:
    d = json.loads(jsons[-1].read_text())
    v = d["verdict"]
    print(f"Final: {v['lat']:.6f}, {v['lng']:.6f}")
    print(f"Confidence: {v['confidence']}")
    print(f"Google Maps: https://www.google.com/maps?q={v['lat']},{v['lng']}")
    if v.get('street_view_url'):
        print(f"Street View: {v['street_view_url']}")